# 06 · Aşama 7 — TinyML: TFLite INT8 dönüşümü
* Merkezi AE (tohum 0) ve FedAvg-IID AE → TFLite **float32** ve **INT8** (temsilî veri: normal-eğitim).
* Raporlanan: boyut, parametre, tahmini RAM, yorumlayıcı gecikmesi (**dizüstü CPU; ESP32 ölçümü değildir**),
  FP32↔INT8 skor uyumu.
* **Güven sorusu (S2):** FP32'de hesaplanan eşikler INT8 skorlarında hâlâ geçerli mi? INT8 ile yeniden kalibrasyon ne değiştirir?

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
import tensorflow as tf, time
from scipy import stats as st
from trustids import data as D, models as M, conformal as C
P = D.load_prepared(ART / "prepared"); X = P["X"]
TINY = ART / "tiny"; TINY.mkdir(exist_ok=True)
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_phase7_tinyml")
SRC = {"Merkezi AE (tohum 0)": ART / "models" / "ae_seed0.keras",
       "FedAvg-IID AE": ART / "fl" / f"fedavg_IID_seed{cfg.train_seeds[0]}.keras"}
rep = X["benign_train"][np.random.default_rng(0).choice(len(X["benign_train"]), 500, replace=False)].astype(np.float32)
def convert(m, int8):
    cv = tf.lite.TFLiteConverter.from_keras_model(m)
    if int8:
        cv.optimizations = [tf.lite.Optimize.DEFAULT]
        cv.representative_dataset = lambda: ([r[None, :]] for r in rep)
        cv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    return cv.convert()
def tfl_score(blob, Xs):
    it = tf.lite.Interpreter(model_content=blob); inp = it.get_input_details()[0]; out = it.get_output_details()[0]
    it.resize_tensor_input(inp["index"], [len(Xs), Xs.shape[1]]); it.allocate_tensors()
    it.set_tensor(inp["index"], Xs.astype(np.float32)); it.invoke(); R = it.get_tensor(out["index"])
    return np.mean((Xs.astype(np.float64) - R.astype(np.float64)) ** 2, axis=1)
def latency_us(blob, x, n=2000):
    it = tf.lite.Interpreter(model_content=blob); inp = it.get_input_details()[0]; it.allocate_tensors()
    it.set_tensor(inp["index"], x[None, :].astype(np.float32)); t0 = time.perf_counter()
    for _ in range(n): it.invoke()
    return (time.perf_counter() - t0) / n * 1e6

In [ ]:
rows, srow = [], []
for name, path in SRC.items():
    if not Path(path).exists(): print("bulunamadı:", path); continue
    m = tf.keras.models.load_model(path)
    fp = convert(m, False); q = convert(m, True)
    tag = "central" if name.startswith("Merkezi") else "fedavg"
    (TINY / f"{tag}_fp32.tflite").write_bytes(fp); (TINY / f"{tag}_int8.tflite").write_bytes(q)
    n_par = M.count_parameters(m); acts = sum(int(l.units) for l in m.layers if hasattr(l, "units"))
    S32 = {sp: M.ae_score(m, X[sp]) for sp in ["benign_cal", "benign_test", "heldout_test", "dev_test"]}
    S8 = {sp: tfl_score(q, X[sp]) for sp in S32}
    np.savez_compressed(TINY / f"{tag}_int8_scores.npz", **S8)
    rows.append({"model": name, "parametre": n_par, "keras_KB": Path(path).stat().st_size / 1024,
                 "tflite_fp32_KB": len(fp) / 1024, "tflite_int8_KB": len(q) / 1024,
                 "tahmini_RAM_KB (ağırlık+aktivasyon, int8)": (n_par + acts) / 1024,
                 "gecikme_fp32_µs": latency_us(fp, X["benign_test"][0]), "gecikme_int8_µs": latency_us(q, X["benign_test"][0]),
                 "Spearman_FP32_INT8": st.spearmanr(S32["benign_test"], S8["benign_test"]).correlation})
    th32 = C.thresholds(S32["benign_cal"], cfg.alpha, cfg.beta)
    for mode, sc, th in [("FP32 skor + FP32 eşik", S32, th32),
                         ("INT8 skor + devralınan FP32 eşik", S8, th32),
                         ("INT8 skor + INT8 yeniden kalibrasyon", S8, None)]:
        if th is None:
            pb = C.pvalues(sc["benign_cal"], sc["benign_test"], seed=0, model="int8"); ph = C.pvalues(sc["benign_cal"], sc["heldout_test"], seed=0, model="int8")
            fpr = np.mean(pb <= cfg.alpha); ha = np.mean(ph <= cfg.alpha); hu = np.mean((ph > cfg.alpha) & (ph <= cfg.alpha + cfg.beta))
        else:
            fpr = np.mean(sc["benign_test"] > th["tau_H"]); ha = np.mean(sc["heldout_test"] > th["tau_H"])
            hu = np.mean((sc["heldout_test"] > th["tau_L"]) & (sc["heldout_test"] <= th["tau_H"]))
        srow.append({"model": name, "kural": mode, "normal_FPR": fpr, "fark_α": fpr - cfg.alpha, "görülmemiş_ANOMALY": ha, "görülmemiş_UNCERTAIN": hu,
                     })
    print(name, "tamam")
TIN = pd.DataFrame(rows); TIN.to_csv(RUN / "06_tinyml_resources.csv", index=False)
S2 = pd.DataFrame(srow); S2.to_csv(RUN / "06_int8_threshold_validity.csv", index=False)
display(TIN.round(3)); S2.round(4)

In [ ]:
cfg.save(RUN / "config_used.json"); print("sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")